**[🏠 Course Home](00_START_HERE.ipynb)** | **🐍 [Python](../python/12_bayesian_regression_and_glms.ipynb)** | **🔵 [R](../r/12_bayesian_regression_and_glms.ipynb)** | **⬅️ [Prev: Ch 11: Two-Group t-Test (Freq vs. Bayes)](11_t_test_frequentist_vs_bayesian.ipynb)** | **➡️ [Next: Ch 13: Hierarchical Models & Shrinkage](13_hierarchical_models_and_partial_pooling.ipynb)**

*🧭 Part III: Applied Modeling, Regression, Hierarchy & Model Comparison — Chapter 12 of 22*

---

# Chapter 12: Modeling Relationships — Bayesian Linear Regression, Poisson-Gamma Counts & GLMs
### *The Moving Bell Curve, The Third Conjugate Pillar, Centered Geometry,*
### *and The Two Uncertainty Ribbons*

---

## 1. Course Context: Moving Beyond Static Group Means to Conditional Models

In earlier chapters, we developed foundational methods to estimate single,
static quantities under uncertainty:
* In **Chapter 1**, we estimated the unknown conversion rate of a single
  button using the Beta-Binomial conjugate update.
* In **Chapter 2**, we weighed sensor precision to determine the true static
  location of a physical asset using Normal-Normal conjugacy.
* In **Chapter 11**, we stepped forward into comparative analysis: asking
  whether Version B was faster than Version A by computing the posterior
  distribution of the difference between two discrete group means $\Delta =
  \mu_B - \mu_A$.

However, real-world physical and software engineering systems rarely operate
at a single static setpoint. Instead, physical systems are dynamic,
continuous, and interdependent:
* **Server Response Latency** is not a static scalar; it changes continuously
  as a function of incoming traffic load (Queries Per Second, or QPS).
* **Network Packet Loss** increases non-linearly as queue depth and buffer
  occupancy approach switch hardware limits.
* **Component Lifetime & Failure Probability** scale sharply with operational
  core temperature, electrical voltage, and clock frequency.

When a server fleet experiences traffic growth from $1,000$ to $5,000$ QPS,
latency does not leap between isolated categorical silos. It moves along a
continuous physical surface.

To model continuous relationships, we replace the static parameter $\mu$ with
a **conditional expectation function**:

$$\mu(x) = \alpha + \beta x$$

Here, $\mu(x)$ is the expected outcome when the external predictor equals $x$.
The parameter $\alpha$ represents the baseline intercept, and $\beta$
represents the rate of change (the slope) per unit increase in $x$.

---

### 🔦 The Physical Analogy: The Moving Spotlight on Rails

To build an intuitive mental model for conditional regression, imagine an
automated inspection gantry mounted above a production conveyor rail:

```text
       THE MOVING BELL CURVE: CONDITIONAL GAUSSIAN REGRESSION

   Latency y (ms)
       ^
       |                                           *  *  (Data scatter)
    70 |                                       *  / \  *
       |                                     *   /   \   *  N(mu(x3), s)
    50 |                      *  *              /=====\
       |                  *  / \  *            *   |
    30 |    *  *         *  /   \   *              |  Mean Line:
       | * / \  *          /=====\                 |  mu(x) = a + b * x
    10 |  /   \  *            |                    |
       | /=====\              |                    |
     0 +----+-----------------+--------------------+---------> Load x (kQPS)
           x1                x2                   x3
```

1. **The Rigid Carriage Rail ($\mu(x) = \alpha + \beta x$)**:
An overhead steel rail runs across the room, sloping upward at angle $\beta$.
   This rail represents the true, expected underlying relationship between
   load $x$ and response time $y$.
2. **The Sliding Spotlight Carriage**:
A motorized carriage slides along the rail as load $x$ varies. At any chosen
   horizontal position $x_i$, the center of the carriage sits at height
   $\mu(x_i)$.
3. **The Bell-Shaped Light Beam (Gaussian Noise $\sigma$)**:
The carriage projects a downward bell-shaped beam of probability. Individual
   physical observations (server requests) do not land directly on the rail.
   Instead, due to concurrent thread scheduling, cache misses, and memory bus
   contention, individual requests scatter around the spotlight focal point
   according to Gaussian spread $\mathcal{N}(0, \sigma^2)$.

Our goal in Bayesian regression is to infer the joint posterior distribution
over the rail's baseline elevation $\alpha$, its slope $\beta$, and the width
of the scatter beam $\sigma$, given a finite set of observed data points
$\{(x_i, y_i)\}_{i=1}^N$.

---

## 2. The Third Conjugate Pillar: Poisson-Gamma for Event Rates ($\lambda$)

Before exploring multi-parameter regression lines, we must complete our
foundational triad of analytical conjugacy.

Throughout this curriculum, we have established two fundamental pillars of
exact Bayesian updating:
1. **Beta-Binomial**: For binary proportions $p \in [0, 1]$ across $n$
   discrete Bernoulli trials (e.g., test passes vs. fails).
2. **Normal-Normal**: For continuous real-valued measurements $y \in
   \mathbb{R}$ with known measurement variance.

The **Third Conjugate Pillar** is the **Poisson-Gamma model**. It governs
**event counts occurring over continuous intervals of time, space, or
volume**.

```text
               THE THREE PILLARS OF EXACT CONJUGACY

   1. Beta-Binomial   --> Binary Trials:       k heads in n flips
   2. Normal-Normal   --> Continuous Values:   y on real line (-inf, +inf)
   3. Poisson-Gamma   --> Continuous Rates:    k events in T time (rate > 0)
```

### The Poisson Likelihood for Event Counts

Consider rare, independent engineering events:
* Kernel panics or node reboots across a distributed cloud cluster.
* Uncorrectable memory ECC errors per terabyte-day.
* Packet drop bursts across an optical backbone link per hour.

If events occur independently at a constant average rate $\lambda > 0$ (events
per unit exposure $T$), the number of observed events $k$ follows a **Poisson
distribution**:

$$P(k \mid \lambda, T) = \frac{(\lambda T)^k e^{-\lambda T}}{k!}$$

Notice that the factor $1/k!$ depends solely on the data count $k$, not on the
unknown rate $\lambda$. Dropping multiplicative constants yields the kernel of
the likelihood:

$$P(k \mid \lambda, T) \propto \lambda^k e^{-\lambda T}$$

---

### The Conjugate Gamma Prior

What prior distribution has a mathematical kernel matching $\lambda^k
e^{-\lambda T}$?
The answer is the **Gamma distribution**, defined for $\lambda > 0$:

$$p(\lambda \mid \alpha_0, \beta_0) =
\frac{\beta_0^{\alpha_0}}{\Gamma(\alpha_0)}
\lambda^{\alpha_0 - 1} e^{-\beta_0 \lambda}
\propto \lambda^{\alpha_0 - 1} e^{-\beta_0 \lambda}$$

Here, $\alpha_0 > 0$ is the **shape parameter** and $\beta_0 > 0$ is the
**rate parameter** (or inverse scale).

The hyperparameters carry an intuitive physical interpretation:
* $\alpha_0$: **Prior pseudo-events** (virtual incidents already observed).
* $\beta_0$: **Prior pseudo-exposure** (virtual hours or node-days elapsed).
* **Prior Mean**: $\mathbb{E}[\lambda] = \frac{\alpha_0}{\beta_0}$
* **Prior Variance**: $\text{Var}(\lambda) = \frac{\alpha_0}{\beta_0^2}$

---

### The Exact Analytical Conjugate Update

When we observe $k$ actual events over exposure time $T$, we apply Bayes' rule
by multiplying prior by likelihood:

$$\begin{aligned}
p(\lambda \mid k, T) &\propto P(k \mid \lambda, T)
\cdot p(\lambda \mid \alpha_0, \beta_0) \\
&\propto \left( \lambda^k e^{-\lambda T} \right)
\cdot \left( \lambda^{\alpha_0 - 1} e^{-\beta_0 \lambda} \right) \\
&= \lambda^{(\alpha_0 + k) - 1} e^{-(\beta_0 + T) \lambda}
\end{aligned}$$

Look closely at the resulting exponent: it is identically another Gamma
distribution!

$$\lambda \mid k, T \sim
\text{Gamma}(\alpha_{\text{post}}, \beta_{\text{post}})$$

$$\alpha_{\text{post}} = \alpha_0 + k, \qquad
\beta_{\text{post}} = \beta_0 + T$$

Just like Beta-Binomial updating, Bayesian updating for event rates requires
nothing more than two additions:
* Add observed events $k$ to the shape parameter $\alpha$.
* Add observed exposure duration $T$ to the rate parameter $\beta$.

The posterior mean is an elegant weighted balance:

$$\mathbb{E}[\lambda \mid k, T] = \frac{\alpha_0 + k}{\beta_0 + T}$$

---

### Step-by-Step Worked Math: Monitoring Fleet Incident Rates

Let us work through a concrete production reliability example:

1. **Prior Belief**:
Historical baseline metrics indicate that healthy cluster nodes suffer
   approximately $0.40$ unrecoverable disk failures per $100$ machine-months.
   We quantify this prior belief with $\alpha_0 = 2.0$ pseudo-failures and
   $\beta_0 = 5.0$ pseudo-hundred-months:
   $$\mathbb{E}[\lambda_0] = \frac{2.0}{5.0} = 0.40
   \text{ failures per unit exposure}$$
   $$\text{Var}(\lambda_0) = \frac{2.0}{5.0^2} = 0.080,
   \quad \sigma_0 = \sqrt{0.080} \approx 0.283$$

2. **Canary Fleet Observation**:
We deploy new storage controller microcode to a canary fleet. Over $T_1 =
   10.0$ units of exposure time, telemetry records $k_1 = 4$ disk failures.

3. **Posterior Calculation**:
   $$\alpha_{\text{post}} = 2.0 + 4 = 6.0$$
   $$\beta_{\text{post}} = 5.0 + 10.0 = 15.0$$
   $$\mathbb{E}[\lambda \mid \text{canary}] = \frac{6.0}{15.0} = 0.400
   \text{ failures per unit exposure}$$
   $$\text{Var}(\lambda \mid \text{canary}) = \frac{6.0}{15.0^2}
   \approx 0.0267, \quad \sigma_{\text{post}} = \sqrt{0.0267} \approx 0.163$$

4. **Information Contraction**:
Notice what happened: while the expected rate remained at $0.40$, the standard
   deviation collapsed from $0.283$ to $0.163$ (a $42\%$ reduction in
   uncertainty). As exposure time $T$ increases, $\text{Var}(\lambda) \approx
   \frac{\lambda}{T} \to 0$.

```text
            POSTERIOR CONCENTRATION WITH ACCUMULATING EXPOSURE

   Density
      ^
      |                                    | Posterior: T = 100h
      |                                  / | \
      |                                 /  |  \
      |               Prior (T = 5h)   /   |   \
      |                . - - - .      /    |    \
      |              /           \   /     |     \
      |             /             \ /      |      \
      |            /               *       |       \
      0 ----------+----------------+-------+--------+----------> Rate lambda
                 0.0              0.4     0.4      1.0
```

---

## 3. Bayesian Linear Regression & The Geometry of Centering

Now let us turn to continuous relationships between predictors and responses.

We formulate the standard Bayesian linear regression model:

$$y_i \sim \mathcal{N}(\mu_i, \sigma), \qquad
\mu_i = \alpha + \beta(x_i - \bar{x})$$

Priors are placed on all unknown parameters:
* Intercept: $\alpha \sim \mathcal{N}(\mu_\alpha, \sigma_\alpha)$
* Slope: $\beta \sim \mathcal{N}(0, \sigma_\beta)$
* Observation Noise: $\ln \sigma \sim \mathcal{N}(\mu_{\ln\sigma},
  \sigma_{\ln\sigma})$

---

### The Danger of Uncentered Predictors: Correlated Valleys

Why do we write $\mu_i = \alpha + \beta(x_i - \bar{x})$ rather than the
elementary textbook formula $\mu_i = \alpha_{\text{raw}} + \beta x_i$?

Suppose we measure server response latency against incoming QPS load, where $x
\in [2000, 6000]$ requests per second. If we leave $x$ uncentered, the
parameter $\alpha_{\text{raw}}$ represents the extrapolated server latency at
**zero QPS** ($x = 0$), an operating regime far outside the observed data!

To fit the observed data points clustered around $\bar{x} \approx 4000$, the
regression line must pass through the centroid $(\bar{x}, \bar{y})$.
Algebraically:

$$\bar{y} \approx \alpha_{\text{raw}} + \beta \bar{x}
\implies \alpha_{\text{raw}} \approx \bar{y} - \beta \bar{x}$$

Consider what happens if our estimate of the slope $\beta$ tilts upward:
Because $\bar{x} = 4000$ is large, increasing $\beta$ by just $0.01$ forces
$\alpha_{\text{raw}}$ to plunge downward by $0.01 \times 4000 = 40\text{ ms}$!

In parameter space, $\alpha_{\text{raw}}$ and $\beta$ become **almost
perfectly negatively correlated** ($r \approx -0.999$). The joint posterior
collapses into a razor-thin, diagonal canyon:

```text
       UNCENTERED (Severe Correlation)         CENTERED (Orthogonal Hessian)
   alpha_raw                              alpha
      ^                                      ^
      |  \  (Narrow diagonal canyon)         |       .---.
      |   \                                  |      /     \  (Decoupled)
      |    \   Optimizers zig-zag,           |     |   +   |
      |     \  MCMC chains stall             |      \     /  Hessian diagonal
      |      \                               |       '---'
      +-------------------------> beta       +-------------------------> beta
```

---

### Centering Diagonalizes the Hessian Matrix

When we center the predictor by subtracting its sample mean, $x_{c, i} = x_i -
\bar{x}$, the sum of the centered predictors is zero:

$$\sum_{i=1}^N (x_i - \bar{x}) = 0$$

Let us examine the second-order partial derivative (the cross-term in the
Hessian matrix of the negative log-likelihood with respect to $\alpha$ and
$\beta$):

$$\frac{\partial \ln L}{\partial \alpha} =
\frac{1}{\sigma^2} \sum_{i=1}^N (y_i - \alpha - \beta x_{c, i})$$

$$\frac{\partial^2 \ln L}{\partial \alpha \, \partial \beta} =
-\frac{1}{\sigma^2} \sum_{i=1}^N x_{c, i} =
-\frac{1}{\sigma^2} \sum_{i=1}^N (x_i - \bar{x}) = 0$$

The cross-partial derivative vanishes completely!

Because the off-diagonal elements of the Hessian matrix are zero, the
parameters $\alpha$ and $\beta$ are **orthogonalized and decoupled**:
1. The parameter $\alpha$ has an immediate physical meaning: it is the
   expected latency at the **average operating load** $\bar{x}$.
2. The parameter $\beta$ is the rate of latency increase per unit load.
3. The posterior covariance between $\alpha$ and $\beta$ is zero:
   $\text{Cov}(\alpha, \beta) = 0$.
4. Numerical optimizers (BFGS, Newton-Raphson) and Hamiltonian MCMC samplers
   converge in a fraction of the iterations without zig-zagging across
   ill-conditioned valleys.

---

## 4. The Two Uncertainty Ribbons: Mean Line vs. Individual Observations

Once we obtain joint posterior samples $(\alpha^{(s)}, \beta^{(s)},
\sigma^{(s)})$ for $s = 1, \dots, S$, how do we visualize uncertainty along
the regression line?

One of the most consequential errors in reliability engineering is confusing
**the uncertainty of the expected relationship** with **the uncertainty of
individual physical measurements**.

Bayesian regression cleanly separates these into **Two Uncertainty Ribbons**:

```text
               THE TWO DISTINCT UNCERTAINTY RIBBONS

   Response y
       ^
       |       / /========================================/ /
       |      / /  Outer Ribbon: 95% Posterior Predictive / / (Points)
       |     / /      (Epistemic Line Uncertainty +       / /
       |    / /        Aleatoric Physical Noise sigma)   / /
       |   / /               .----------------.         / /
       |  / /               / Inner Ribbon:   /        / /
       | / /               / 95% Credible    /        / /
       |/ /               / Mean Line mu(x) /        / /
       | /               '----------------'         / /
       |/==========================================/ /
       +----------------------------------------------------------> x
                                 x_bar
```

---

### Ribbon 1: The 95% Credible Band of the Mean Line $\mu(x)$

For any specific value of the predictor $x_*$, the expected response across
posterior sample $s$ is:

$$\mu^{(s)}(x_*) = \alpha^{(s)} + \beta^{(s)}(x_* - \bar{x})$$

By computing the $2.5\%$ and $97.5\%$ percentiles of $\mu^{(s)}(x_*)$ across
all draws $s = 1, \dots, S$, we trace out the **$95\%$ Credible Ribbon of the
Mean Line**.

* **What it represents**: Our **epistemic uncertainty** about the true,
  unobservable average response line.
* **Shape**: It forms an elegant **hyperbolic hourglass**. It is narrowest at
  the data center $\bar{x}$ (where slope uncertainty has zero leverage:
  $\text{Var}(\mu(\bar{x})) = \text{Var}(\alpha)$). It fans outward like a
  trumpet as $|x_* - \bar{x}|$ grows because small uncertainties in slope
  $\beta$ multiply across distance.
* **Asymptotic Behavior**: If we collect infinite data ($N \to \infty$),
  parameter uncertainty shrinks to zero: the mean line credible ribbon
  **collapses into a razor-thin line**.

---

### Ribbon 2: The 95% Posterior Predictive Band for Points $y_{\text{new}}$

In production operations, an engineer rarely cares solely about the fleet
average. When an end-user sends an individual query to our service, that query
experiences a specific, individual latency $y_{\text{new}}$.

To compute the predictive distribution, we simulate a synthetic observation
for every posterior draw $s$:

$$y_{\text{new}}^{(s)}(x_*) =
\mathcal{N}(\mu^{(s)}(x_*), \sigma^{(s)})$$

By calculating the $2.5\%$ and $97.5\%$ percentiles of
$y_{\text{new}}^{(s)}(x_*)$, we trace out the **$95\%$ Posterior Predictive
Ribbon**.

* **What it represents**: The complete range of where future individual
  physical observations will fall.
* **Mathematical Decomposition of Variance**:
  $$\text{Var}(y_{\text{new}} \mid x_*) =
  \underbrace{\text{Var}(\mu(x_*))}_{\text{Epistemic (Parameter)}} +
  \underbrace{\mathbb{E}[\sigma^2]}_{\text{Aleatoric (Physical Noise)}}$$
* **Asymptotic Behavior**: Even with infinite data ($N \to \infty$), this
  ribbon **never collapses**! Parameter uncertainty vanishes, but the
  irreducible physical noise $\sigma$ remains.

Conflating these two ribbons leads to catastrophic decisions: using the narrow
credible ribbon to set production Service Level Objectives (SLOs) guarantees
constant alert violations, because individual user queries scatter across the
much wider predictive ribbon!

---

## 5. Generalized Linear Models (GLMs): Bayesian Logistic Regression

What happens when our response variable is **not continuous and Gaussian**,
but **binary** ($y_i \in \{0, 1\}$)?
* Did a hardware node pass or fail a stress test at operating temperature $T$?
* Did a distributed transaction commit ($1$) or abort ($0$)?
* Did a network switch drop a packet frame ($1$) or forward it ($0$)?

---

### Why Linear Regression Fails for Binary Outcomes

Attempting to fit a standard linear line $p(x) = \alpha + \beta x$ to binary
data fails fundamentally on two mathematical grounds:

1. **Probability Bound Violations**:
A straight line has domain and range $(-\infty, \infty)$. At extreme
   temperatures, the linear line will inevitably predict $p < 0$ or $p > 1$,
   producing mathematically impossible probabilities.
2. **Heteroskedastic Non-Gaussian Variance**:
For binary data $y \sim \text{Bernoulli}(p)$, the variance is strictly a
   function of the mean: $\text{Var}(y) = p(1 - p)$. Variance is zero at $p=0$
   and $p=1$, and peaks at $p=0.5$. Constant residual variance $\sigma^2$ is
   physically impossible.

```text
       THE FAILURE OF LINEAR REGRESSION ON PROBABILITIES
   Probability p
       ^
   1.2 |                              /  Linear Line: Predicts p > 1.0!
   1.0 |-----------------------------/---------------- (Physical Maximum)
       |                            /     . - - ' S-Curve Sigmoid
   0.5 |                           / . - '        Keeps p in (0, 1)
       |                     . - '/
   0.0 |-------------------/-------------------------- (Physical Minimum)
  -0.2 |                  /  Linear Line: Predicts p < 0.0!
       +-----------------+---------------+------------> Temperature x
```

---

### The GLM Solution: The Logit Link Function

To resolve this dilemma, Nelder and Wedderburn (1972) introduced **Generalized
Linear Models (GLMs)**. A GLM consists of three parts:
1. **Random Component**: Likelihood distribution (e.g.,
   $\text{Bernoulli}(p)$).
2. **Systematic Component**: Linear predictor $\eta = \alpha + \beta(x -
   \bar{x})$.
3. **Link Function $g(p)$**: Monotonic mapping from probability $p \in (0, 1)$
   to unbounded linear predictor $\eta \in (-\infty, \infty)$.

For binary outcomes, the canonical link is the **Logit link** (log-odds):

$$\eta = g(p) = \ln \left( \frac{p}{1 - p} \right) =
\alpha + \beta(x - \bar{x})$$

To compute the expected probability $p$, we invert the link function using the
**Logistic Sigmoid** (inverse-logit):

$$p = g^{-1}(\eta) = \frac{1}{1 + e^{-\eta}} =
\frac{1}{1 + e^{-(\alpha + \beta(x - \bar{x}))}}$$

The sigmoid function maps the entire real line $(-\infty, \infty)$ smoothly
and monotonically into the valid probability interval $(0, 1)$.

---

### Bayesian Inference on the S-Curve: Certifying Safe Operating Envelopes

In a Bayesian GLM, our parameters $(\alpha, \beta)$ have a joint posterior
distribution. Every sampled pair $(\alpha^{(s)}, \beta^{(s)})$ produces a
distinct S-curve across temperature:

```text
            BAYESIAN ENSEMBLE OF LOGISTIC S-CURVES

   Failure Prob p
       ^
   1.0 |                                       . - - '
       |                                   . -'  /  /
       |                                . '    /  /
   0.5 |                             . '     /  /
       |                          . '      /  /
  0.05 | - - - - - - - - - - - * '       /  /
       |                   * ' |       /  /
   0.0 |_________________*'____|_____/__/________________> Temperature
                        T_safe  T_med
```

By pushing posterior parameter draws through the sigmoid function, we obtain a
posterior distribution over failure probability at every temperature $T$.

This provides a principled solution to mission-critical engineering design:
> **Question**: *At what maximum temperature can we certify a server fleet if
> our contractual reliability policy dictates that the failure probability
> must not exceed $5\%$, with $95\%$ confidence?*

Rather than relying on a fragile point estimate (which would expose us to a
$50\%$ chance of underestimating risk), Bayesian GLMs allow us to find the
temperature $T_{\text{safe}}$ where the **$95\text{th}$ percentile of the
posterior failure probability curve** crosses $0.05$.

---

## 6. Chapter Summary & Comparison Matrix

| Model Family | Target Domain | Parameters | Inference Engine |
| :--- | :--- | :--- | :--- |
| **Poisson** | Events in time $T$ | $\lambda > 0$ | Exact Conjugate |
| **Linear** | Continuous $y \in \mathbb{R}$ | $\alpha, \beta, \sigma$ | Laplace / MCMC |
| **Logistic** | Binary $y \in \{0, 1\}$ | $\alpha, \beta$ | Laplace / MCMC |

Now, let us bring these mathematical models to life with complete, runnable
implementations in our accompanying computational sheets:
* 🐍 **Python Track**: [Chapter 12: Regression &
  GLMs](../python/12_bayesian_regression_and_glms.ipynb)
* 📊 **R Track**: [Chapter 12: Regression &
  GLMs](../r/12_bayesian_regression_and_glms.ipynb)

---
## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 11: Two-Group t-Test (Freq vs. Bayes)](11_t_test_frequentist_vs_bayesian.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* ➡️ **Next**: **[Ch 13: Hierarchical Models & Shrinkage](13_hierarchical_models_and_partial_pooling.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* 🔄 **Companion Tracks for Chapter 12**: [📖 Conceptual](../conceptual/12_bayesian_regression_and_glms.ipynb) | [🐍 Python](../python/12_bayesian_regression_and_glms.ipynb) | [🔵 R](../r/12_bayesian_regression_and_glms.ipynb)
